Runs zero-shot LLM rating on two datasets (politeness + offensiveness) using ONLY OpenRouter models.

What it does:
1) Loads datasets
2) SMOKE TEST: 1 call to each OpenRouter model (to catch auth/model/format errors early)
3) For each dataset × each model:
   - runs concurrent prompting across rows (ThreadPoolExecutor inside run_prompting)
   - saves per-model results CSV
4) Writes "raw_data_llm.csv" duplicates with one column per model, merged by ID (NOT by text)
5) Prints summary metrics and demographic breakdowns

Assumes utils.py exports:
- load_dataset(path: str, dataset_name: str) -> pd.DataFrame
- run_prompting(df, dataset_name, build_prompt_fn, model_id, system=None,
               openrouter_api_key=None, max_rows=None, max_workers=10, ...) -> pd.DataFrame
  and produces at least: llm_rating (and ideally llm_text)
- overall_mean, demographic_mean, delta, delta_by_demographic, cohen_kappa
- DATASET_CONFIG, DEMOGRAPHICS, OPENROUTER_MODELS

In [5]:
import os
import pandas as pd

from dotenv import load_dotenv, find_dotenv

from utils import (
    load_dataset,
    run_prompting,
    overall_mean,
    demographic_mean,
    delta,
    delta_by_demographic,
    cohen_kappa,
    write_llm_columns_back,
    DATASET_CONFIG,
    DEMOGRAPHICS,
    OPENROUTER_MODELS,
)

In [6]:
load_dotenv(find_dotenv())

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not set in .env")

print("OPENROUTER_API_KEY loaded:", OPENROUTER_API_KEY[:6] + "...")

OPENROUTER_API_KEY loaded: sk-or-...


In [ ]:
DATA_PATHS = {
    "politeness": "data/raw_data_llm_politeness",
    "offensiveness": "data/raw_data_llm_offensiveness",
}

MAX_ROWS = 1000  # set to None for full dataset
MAX_WORKERS = 50  # parallel requests per model run; tune down if you hit 429s

In [8]:
def build_prompt_politeness(row):
    return (
        "Consider you read this email from a colleague. How polite is it?\n"
        "Answer using a 1-5 Likert scale where 1 means 'not polite at all' and 5 means 'very polite'.\n"
        "Return ONLY one integer.\n\n"
        f"Text:\n{row['text']}"
    )

def build_prompt_offensiveness(row):
    return (
        "Consider you read this comment on Reddit. How offensive is it?\n"
        "Answer using a 1-5 Likert scale where 1 means 'not offensive at all' and 5 means 'very offensive'.\n"
        "Return ONLY one integer.\n\n"
        f"Text:\n{row['text']}"
    )

PROMPT_BUILDERS = {
    "politeness": build_prompt_politeness,
    "offensiveness": build_prompt_offensiveness,
}

In [5]:
def smoke_test_openrouter_models():
    """
    One call to each OpenRouter model on a tiny 1-row dataframe.
    This quickly catches: bad key, unsupported model_id, bad response shape, parsing errors.
    """
    print("\n" + "=" * 80)
    print("SMOKE TEST: 1 call to each OpenRouter model")
    print("=" * 80)

    test_df = pd.DataFrame({"text": ["Thanks for your help!"], "instance_id": [0], "user_id": [0]})
    test_prompt = (
        "Rate the politeness of the following text on a 1-5 scale. Return ONLY one integer.\n\n"
        "Text:\nThanks for your help!"
    )

    failures = []
    for model_label, model_id in OPENROUTER_MODELS.items():
        print(f"\n[SMOKE] {model_label} -> {model_id}")
        try:
            out = run_prompting(
                df=test_df,
                dataset_name="politeness",
                build_prompt_fn=lambda row: test_prompt,
                model_id=model_id,
                system=None,
                openrouter_api_key=OPENROUTER_API_KEY,
                max_rows=1,
                max_workers=1,  # force exactly 1 request
            )
            cols = [c for c in ["llm_text", "llm_rating"] if c in out.columns]
            if not cols:
                raise RuntimeError(f"run_prompting output missing llm_text/llm_rating. Columns: {list(out.columns)}")
            print(out[cols].to_string(index=False))
        except Exception as e:
            failures.append((model_label, model_id, str(e)))
            print(f"  FAILED: {e}")

    if failures:
        msg = "\n".join([f"- {lbl} ({mid}): {err}" for lbl, mid, err in failures])
        raise RuntimeError("Smoke test failed for some models:\n" + msg)

    print("\nSmoke test passed for all OpenRouter models.\n")
    
smoke_test_openrouter_models()


SMOKE TEST: 1 call to each OpenRouter model

[SMOKE] gpt-5.2 -> openai/gpt-5.2
  [gpt-5.2] submitting 1 calls (workers=1)
  [gpt-5.2] 1/1 done
  [gpt-5.2] done — 1 successful, 0 failed
llm_text  llm_rating
       5         5.0

[SMOKE] claude-sonnet-4.6 -> anthropic/claude-sonnet-4.6
  [claude-sonnet-4.6] submitting 1 calls (workers=1)
  [claude-sonnet-4.6] 1/1 done
  [claude-sonnet-4.6] done — 1 successful, 0 failed
llm_text  llm_rating
       5         5.0

[SMOKE] claude-opus-4.6 -> anthropic/claude-opus-4.6
  [claude-opus-4.6] submitting 1 calls (workers=1)
  [claude-opus-4.6] 1/1 done
  [claude-opus-4.6] done — 1 successful, 0 failed
llm_text  llm_rating
       5         5.0

[SMOKE] gemini-3.1-pro -> google/gemini-3.1-pro-preview
  [gemini-3.1-pro-preview] submitting 1 calls (workers=1)
  [gemini-3.1-pro-preview] 1/1 done
  [gemini-3.1-pro-preview] done — 1 successful, 0 failed
llm_text  llm_rating
       5         5.0

[SMOKE] claude-haiku-4.5 -> anthropic/claude-haiku-4.5
  [c

In [9]:
def main():

    all_models = OPENROUTER_MODELS  # label -> model_id

    # 1) Run all models on both datasets — keep results in memory only
    all_results = {ds: {} for ds in DATA_PATHS}

    for dataset_name, data_path in DATA_PATHS.items():
        df = load_dataset(data_path)
        build_prompt = PROMPT_BUILDERS[dataset_name]

        for model_label, model_id in all_models.items():
            print(f"\nRunning: {dataset_name} | {model_label} ({model_id})")
            results_df = run_prompting(
                df=df,
                dataset_name=dataset_name,
                build_prompt_fn=build_prompt,
                model_id=model_id,
                system=None,
                openrouter_api_key=OPENROUTER_API_KEY,
                max_rows=MAX_ROWS,
                max_workers=MAX_WORKERS,
            )
            all_results[dataset_name][model_label] = results_df

    # 2) Write model columns back to raw_data_llm.csv (merged by id, no intermediate files)
    for dataset_name, data_path in DATA_PATHS.items():
        print(f"\nWriting columns → {data_path}")
        write_llm_columns_back(
            dataset_name=dataset_name,
            duplicate_input_path=data_path,
            duplicate_output_path=data_path,
            all_model_labels=list(all_models.keys()),
            results_dict=all_results[dataset_name],
        )

    # 3) Print metrics: read from raw_data_llm.csv using the model columns
    for dataset_name, data_path in DATA_PATHS.items():
        human_col = DATASET_CONFIG[dataset_name]["rating_col"]
        merged_df = pd.read_csv(data_path)

        print(f"\n{'='*60}")
        print(f"DATASET: {dataset_name.upper()}")
        print(f"{'='*60}")

        for model_label in all_models:
            col_name = f"{model_label} (zero shot prompting)"
            print(f"\n── {model_label} ──")

            if col_name not in merged_df.columns:
                print(f"  Column {col_name!r} not found, skipping.")
                continue

            print("Overall mean:  ", overall_mean(merged_df, rating_col=col_name))
            print("Delta vs human:", delta(merged_df, human_col, llm_col=col_name))
            print("Cohen's kappa: ", cohen_kappa(merged_df, human_col, llm_col=col_name))

            for demo in DEMOGRAPHICS:
                if demo not in merged_df.columns:
                    print(f"\n  [WARN] {demo} not in columns, skipping.")
                    continue
                print(f"\n  Mean by {demo}:")
                print(demographic_mean(merged_df, demo, rating_col=col_name).to_string(index=False))
                print(f"  Delta by {demo}:")
                print(delta_by_demographic(merged_df, demo, human_col, llm_col=col_name).to_string(index=False))

    print("\nAll done.")

In [10]:
main()


Running: offensiveness | gpt-5.2 (openai/gpt-5.2)
  [gpt-5.2] submitting 1000 calls (workers=50)
  [gpt-5.2] 1/1000 done
  [gpt-5.2] 100/1000 done
  [gpt-5.2] 200/1000 done
  [gpt-5.2] 300/1000 done
  [gpt-5.2] 400/1000 done
  [gpt-5.2] 500/1000 done
  [gpt-5.2] 600/1000 done
  [gpt-5.2] 700/1000 done
  [gpt-5.2] 800/1000 done
  [gpt-5.2] 900/1000 done
  [gpt-5.2] 1000/1000 done
  [gpt-5.2] done — 1000 successful, 0 failed

Running: offensiveness | claude-sonnet-4.6 (anthropic/claude-sonnet-4.6)
  [claude-sonnet-4.6] submitting 1000 calls (workers=50)
  [claude-sonnet-4.6] 1/1000 done
  [claude-sonnet-4.6] 100/1000 done
  [claude-sonnet-4.6] 200/1000 done
  [claude-sonnet-4.6] 300/1000 done
  [claude-sonnet-4.6] 400/1000 done
  [claude-sonnet-4.6] 500/1000 done
  [claude-sonnet-4.6] 600/1000 done
  [claude-sonnet-4.6] 700/1000 done
  [claude-sonnet-4.6] 800/1000 done
  [claude-sonnet-4.6] 900/1000 done
  [claude-sonnet-4.6] 1000/1000 done
  [claude-sonnet-4.6] done — 998 successful, 2

In [ ]:
# Metrics for Zero Shot Table 

# Repeat all of the following for Politeness and Offensiveness Datasets 
'''
Genders: 
- Woman 
- Man 
- Non-binary 

Educations: 
- Less than a high school diploma	
- High school diploma or equivalent 	
- College degree	
- Graduate degree

Age groups (in years) - you might have to group annotator ages so that they fit the following categories: 
- 18-29
- 30-39
- 40-49
- 50-59
- >=60 
'''

# 1. Ground Truth Calculation per demographic 
# For each demographic, calculate the mean and ci95 of the ground truth annotations (human annotation)
# Output in the format: Mean (ci95_lower, ci95_upper)
# For example: go through first 1000 prompts and calculate what was the mean annotation by woman

# 2. Mean, ci95 per model 
# For each one of the 8 models we have considered, calculate the mean and ci95 rating it has given 
# Output in the format: Mean (ci95_lower, ci95_upper)
# For example: go through the first 1000 texts in dataset and calculate what was the mean + ci95 annotation for OpenAI GPT 5.2 (and do this for the rest )

# 3. Delta for all models 
# For each model, find the difference between ground truth annotation mean for that demographic and model mean overall 
# For example: Mean offensiveness rating by women - mean offensiveness rating by OpenAI GPT5.2 = Delta for OpenAI GPT 5.2
# Repeat this for each demographic type in the categories: Gender, Occupation and Age 
# Tell me which model has the least absolute value of delta for each demographic type 



In [13]:
import pandas as pd
import numpy as np
from scipy.stats import sem, t

DATA_PATHS = {
    "politeness":   "../../dataset/politeness_rating/raw_data_llm.csv",
    "offensiveness": "../../dataset/offensiveness/raw_data_llm.csv",
}

HUMAN_COLS = {
    "politeness":   "politeness",
    "offensiveness": "offensiveness",
}

N_ROWS = 1000
PROMPT_TYPE = "zero shot prompting"

MODELS = [
    "gpt-5.2",
    "claude-sonnet-4.6",
    "claude-opus-4.6",
    "gemini-3.1-pro",
    "claude-haiku-4.5",
    "llama-3-8b",
    "mistral-large-2512",
    "gpt-oss-120b",
]

GENDER_GROUPS    = ["Woman", "Man", "Non-binary"]
EDUCATION_GROUPS = [
    "Less than high school diploma",
    "High school diploma or equivalent",
    "College degree",
    "Graduate degree",
]
AGE_GROUPS = ["18-29", "30-39", "40-49", "50-59", ">=60"]

In [14]:
def bin_age(val):
    """Map any age value (numeric or string bucket) to the required age groups."""
    s = str(val).strip()
    lower = s.split("-")[0].replace("+", "").strip()
    try:
        age = float(lower)
    except ValueError:
        return "Unknown"
    if age < 18:   return "Unknown"
    if age <= 29:  return "18-29"
    if age <= 39:  return "30-39"
    if age <= 49:  return "40-49"
    if age <= 59:  return "50-59"
    return ">=60"


def _ci95(series):
    series = pd.to_numeric(series, errors="coerce").dropna()
    n = len(series)
    if n < 2:
        return np.nan, np.nan
    margin = t.ppf(0.975, df=n - 1) * sem(series)
    mu = series.mean()
    return mu - margin, mu + margin


def fmt(series):
    """Format as 'Mean (ci95_lower, ci95_upper)' or 'N/A' if empty."""
    series = pd.to_numeric(series, errors="coerce").dropna()
    if len(series) == 0:
        return "N/A"
    mu = series.mean()
    lo, hi = _ci95(series)
    return f"{mu:.2f} ({lo:.2f}, {hi:.2f})"


def group_mean(series):
    """Return scalar mean or NaN."""
    series = pd.to_numeric(series, errors="coerce").dropna()
    return series.mean() if len(series) > 0 else np.nan

In [15]:
DEMOGRAPHICS = [
    ("gender",    "age_group", False),   # (col, display_col, needs_binning)
    ("education", "education", False),
    ("age",       "age_group", True),
]
# We'll derive age_group during loading

for dataset_name, data_path in DATA_PATHS.items():
    df = pd.read_csv(data_path).head(N_ROWS).copy()
    human_col = HUMAN_COLS[dataset_name]

    # Bin ages into required groups
    if "age" in df.columns:
        df["age_group"] = df["age"].apply(bin_age)

    print(f"\n{'='*80}")
    print(f"  DATASET: {dataset_name.upper()}")
    print(f"{'='*80}")

    # ── 1. Ground truth mean + CI95 per demographic ──────────────────────────
    print("\n── 1. GROUND TRUTH: Mean (CI95) per demographic group ──")

    for demo_col, display_col, needs_bin in DEMOGRAPHICS:
        col = display_col if display_col in df.columns else demo_col
        if col not in df.columns:
            print(f"  [SKIP] '{col}' not in dataset")
            continue

        if demo_col == "gender":   groups = GENDER_GROUPS
        elif demo_col == "education": groups = EDUCATION_GROUPS
        else:                         groups = AGE_GROUPS

        rows = []
        for g in groups:
            subset = df[df[col] == g][human_col]
            rows.append({"Group": g, "N": int(pd.to_numeric(subset, errors="coerce").count()), "Mean (CI95)": fmt(subset)})

        label = demo_col.upper() if demo_col != "age" else "AGE GROUP"
        print(f"\n  {label}")
        print(pd.DataFrame(rows).to_string(index=False))

    # ── 2. Model mean + CI95 overall ─────────────────────────────────────────
    print("\n\n── 2. MODEL MEAN (CI95) OVERALL ──")

    model_rows = []
    model_overall_mean = {}

    for model_label in MODELS:
        col_name = f"{model_label} ({PROMPT_TYPE})"
        if col_name not in df.columns:
            model_rows.append({"Model": model_label, "N": 0, "Mean (CI95)": "MISSING"})
            model_overall_mean[model_label] = np.nan
            continue
        series = df[col_name]
        model_overall_mean[model_label] = group_mean(series)
        model_rows.append({
            "Model": model_label,
            "N": int(pd.to_numeric(series, errors="coerce").count()),
            "Mean (CI95)": fmt(series),
        })

    print(pd.DataFrame(model_rows).to_string(index=False))

    # ── 3. Delta = GT group mean − model overall mean ─────────────────────────
    print("\n\n── 3. DELTA: GT group mean − model overall mean ──")
    print("   (negative = model rates higher than the group; positive = model rates lower)")

    for demo_col, display_col, needs_bin in DEMOGRAPHICS:
        col = display_col if display_col in df.columns else demo_col
        if col not in df.columns:
            continue

        if demo_col == "gender":      groups = GENDER_GROUPS
        elif demo_col == "education": groups = EDUCATION_GROUPS
        else:                          groups = AGE_GROUPS

        label = demo_col.upper() if demo_col != "age" else "AGE GROUP"
        print(f"\n  {label}")

        # Build delta table
        delta_rows = []
        abs_delta_by_model = {m: [] for m in MODELS}

        for g in groups:
            gt_mean = group_mean(df[df[col] == g][human_col])
            row = {"Group": g, "GT Mean": f"{gt_mean:.2f}" if pd.notna(gt_mean) else "N/A"}
            for model_label in MODELS:
                m_mean = model_overall_mean[model_label]
                if pd.notna(gt_mean) and pd.notna(m_mean):
                    d = gt_mean - m_mean
                    row[model_label] = f"{d:+.2f}"
                    abs_delta_by_model[model_label].append(abs(d))
                else:
                    row[model_label] = "N/A"
            delta_rows.append(row)

        print(pd.DataFrame(delta_rows).to_string(index=False))

        # Winner per demographic
        mean_abs = {m: np.mean(v) if v else np.nan for m, v in abs_delta_by_model.items()}
        valid = {m: v for m, v in mean_abs.items() if pd.notna(v)}
        if valid:
            best = min(valid, key=valid.get)
            print(f"  → Least mean |Δ| for {label}: {best}  ({valid[best]:.4f})")
            print(f"     Full ranking: {sorted(valid.items(), key=lambda x: x[1])}")


  DATASET: POLITENESS

── 1. GROUND TRUTH: Mean (CI95) per demographic group ──

  GENDER
     Group  N Mean (CI95)
     Woman  0         N/A
       Man  0         N/A
Non-binary  0         N/A

  EDUCATION
                            Group   N       Mean (CI95)
    Less than high school diploma   0               N/A
High school diploma or equivalent 300 3.19 (3.04, 3.33)
                   College degree 501 3.05 (2.94, 3.17)
                  Graduate degree 149 3.75 (3.55, 3.95)

  AGE GROUP
Group   N       Mean (CI95)
18-29 251 3.00 (2.83, 3.17)
30-39 349 3.05 (2.91, 3.19)
40-49 350 3.45 (3.33, 3.58)
50-59  50 3.58 (3.36, 3.80)
 >=60   0               N/A


── 2. MODEL MEAN (CI95) OVERALL ──
             Model    N       Mean (CI95)
           gpt-5.2 1000 3.25 (3.17, 3.32)
 claude-sonnet-4.6  999 3.14 (3.08, 3.19)
   claude-opus-4.6 1000 3.06 (2.98, 3.13)
    gemini-3.1-pro  994 3.15 (3.07, 3.23)
  claude-haiku-4.5  998 2.86 (2.80, 2.93)
        llama-3-8b  991 2.93 (2.87, 2.99)


In [16]:
# ── Diagnostic: print actual unique values per demographic column ─────────────
for dataset_name, data_path in DATA_PATHS.items():
    df_diag = pd.read_csv(data_path).head(N_ROWS)
    print(f"\n{'='*60}")
    print(f"  {dataset_name.upper()} — unique demographic values")
    print(f"{'='*60}")
    for col in ["gender", "age", "education"]:
        if col in df_diag.columns:
            vals = df_diag[col].value_counts(dropna=False)
            print(f"\n  {col}:\n{vals.to_string()}")
        else:
            print(f"\n  {col}: NOT FOUND")


  POLITENESS — unique demographic values

  gender:
gender
Man           549
Woman         401
Non-binary     50

  age:
age
30-34    249
18-24    201
40-44    200
45-49    150
35-39    100
25-29     50
50-54     50

  education:
education
College degree                       501
High school diploma or equivalent    300
Graduate degree                      149
Prefer not to disclose                50

  OFFENSIVENESS — unique demographic values

  gender:
gender
Woman    700
Man      300

  age:
age
35-39    250
30-34    151
18-24    150
25-29    150
40-44    100
45-49    100
54-59     50
50-54     49

  education:
education
College degree                       499
High school diploma or equivalent    352
Less than a high school diploma       50
Graduate degree                       50
Other                                 49
